In [1]:
import pandas as pd
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

data_folder = project_folder / "data"
feature_file = data_folder / "propertylens_featured.csv"

df = pd.read_csv(feature_file)

# Locality-level summary
locality_summary = (
    df.groupby("locality")
    .agg(
        listings=("locality", "size"),
        avg_price_lakh=("price_lakh", "mean"),
        median_price_lakh=("price_lakh", "median"),
        avg_rate_per_sqft=("rate_per_sqft", "mean"),
        median_rate_per_sqft=("rate_per_sqft", "median"),
        avg_area_sqft=("area_sqft", "mean"),
        min_price_lakh=("price_lakh", "min"),
        max_price_lakh=("price_lakh", "max")
    )
    .reset_index()
)

# Keep localities with enough observations for comparison
locality_summary = locality_summary[locality_summary["listings"] >= 30].copy()

# Dominant property type
dominant_property_type = (
    df.groupby("locality")["flat_type"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index(name="dominant_property_type")
)

# RERA approval share
rera_share = (
    df.assign(
        rera_approved=df["rera_approval"].eq("Approved by RERA")
    )
    .groupby("locality")["rera_approved"]
    .mean()
    .mul(100)
    .reset_index(name="rera_approved_pct")
)

# Ready-to-move share
ready_share = (
    df.assign(
        ready_to_move=df["status"].eq("Ready to move")
    )
    .groupby("locality")["ready_to_move"]
    .mean()
    .mul(100)
    .reset_index(name="ready_to_move_pct")
)

# Combine locality-level information
locality_summary = (
    locality_summary
    .merge(dominant_property_type, on="locality", how="left")
    .merge(rera_share, on="locality", how="left")
    .merge(ready_share, on="locality", how="left")
)

# Top localities by listing volume
print("Top localities by listing volume:")
print(
    locality_summary
    .sort_values("listings", ascending=False)
    [["locality", "listings", "median_price_lakh", "median_rate_per_sqft"]]
    .head(15)
    .to_string(index=False)
)

# Highest median property price
print("\nLocalities with highest median price:")
print(
    locality_summary
    .sort_values("median_price_lakh", ascending=False)
    [["locality", "listings", "median_price_lakh", "avg_price_lakh", "dominant_property_type"]]
    .head(15)
    .to_string(index=False)
)

# Highest median rate per sqft
print("\nLocalities with highest median rate per sqft:")
print(
    locality_summary
    .sort_values("median_rate_per_sqft", ascending=False)
    [["locality", "listings", "median_rate_per_sqft", "avg_rate_per_sqft"]]
    .head(15)
    .to_string(index=False)
)

# Largest average property area
print("\nLocalities with largest average area:")
print(
    locality_summary
    .sort_values("avg_area_sqft", ascending=False)
    [["locality", "listings", "avg_area_sqft", "median_price_lakh"]]
    .head(15)
    .to_string(index=False)
)

# RERA approval share
print("\nLocalities with highest RERA approval share:")
print(
    locality_summary
    .sort_values("rera_approved_pct", ascending=False)
    [["locality", "listings", "rera_approved_pct"]]
    .head(15)
    .to_string(index=False)
)

# Ready-to-move share
print("\nLocalities with highest ready-to-move share:")
print(
    locality_summary
    .sort_values("ready_to_move_pct", ascending=False)
    [["locality", "listings", "ready_to_move_pct"]]
    .head(15)
    .to_string(index=False)
)

Top localities by listing volume:
       locality  listings  median_price_lakh  median_rate_per_sqft
      Sector 65       857              601.0               17980.0
      Sector 79       576              156.0               10879.5
      Sector 63       498              350.0               20253.0
      Sector 67       486              353.5               15569.0
     Sector 113       454              408.0               20266.0
     Sector 108       439              350.0               17196.0
     Sector 102       427              180.0                9751.0
     Sector 37D       421              253.0               11250.0
     Sector 106       418              291.0               16500.0
     Sector 36A       396              395.0               20000.0
      Sector 85       350              225.0               12222.0
      Sector 76       349              685.0               18558.0
      Sector 54       287              808.0               27722.0
     Sector 111       271   